# YOLOv10 + PPLCNet — Oyster Mushroom Detection

This notebook walks through:
1. Environment setup
2. Dataset loading & sanity check
3. Model architecture overview
4. Training
5. Evaluation (validation loss + IoU)
6. Inference & bounding-box visualisation

> **Colab users:** Mount your Drive and unzip the dataset before running.
> ```python
> from google.colab import drive
> drive.mount('/content/drive')
> !unzip /content/drive/MyDrive/oyster_mushroom_yolo.zip -d .
> ```

## 1. Environment Setup

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 2. Dataset

In [ ]:
from dataset import OysterDataset
from torch.utils.data import DataLoader

TRAIN_IMG = 'dataset/train/images'
TRAIN_LBL = 'dataset/train/labels'
VAL_IMG   = 'dataset/val/images'
VAL_LBL   = 'dataset/val/labels'

train_ds = OysterDataset(TRAIN_IMG, TRAIN_LBL)
val_ds   = OysterDataset(VAL_IMG,   VAL_LBL)

print(f'Train samples : {len(train_ds)}')
print(f'Val   samples : {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2)

imgs, labels = next(iter(train_loader))
print(f'Image batch shape : {imgs.shape}')    # (B, 3, 640, 640)
print(f'Label batch shape : {labels.shape}')  # (B, 5)

## 3. Model Architecture

In [ ]:
from pplcnet import PPLCNetDetector

model = PPLCNetDetector(num_classes=1).to(device)

# Sanity-check forward pass
dummy = torch.randn(1, 3, 640, 640).to(device)
out   = model(dummy)
print('Output shape:', out.shape)  # Expected: torch.Size([1, 6])

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## 4. Training

In [ ]:
import torch.optim as optim

model     = PPLCNetDetector(num_classes=1).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

EPOCHS = 20

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        conf   = torch.ones((labels.size(0), 1), device=device)
        labels = torch.cat([labels, conf], dim=1)  # (B, 6)

        preds = model(imgs)
        loss  = criterion(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            conf   = torch.ones((labels.size(0), 1), device=device)
            labels = torch.cat([labels, conf], dim=1)
            val_loss += criterion(model(imgs), labels).item()

    print(f'Epoch [{epoch:>3}/{EPOCHS}]  '
          f'Train Loss: {train_loss/len(train_loader):.6f}  '
          f'Val Loss: {val_loss/len(val_loader):.6f}')

torch.save(model.state_dict(), 'pplcnet_oyster_final.pt')
print('\n✅ Weights saved to pplcnet_oyster_final.pt')

## 5. Evaluation (IoU)

In [ ]:
def compute_iou(pred_box, gt_box):
    """IoU for [cx, cy, w, h] normalised boxes."""
    def corners(cx, cy, w, h):
        return cx - w/2, cy - h/2, cx + w/2, cy + h/2
    px1,py1,px2,py2 = corners(*pred_box)
    gx1,gy1,gx2,gy2 = corners(*gt_box)
    ix1,iy1 = max(px1,gx1), max(py1,gy1)
    ix2,iy2 = min(px2,gx2), min(py2,gy2)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    union = (px2-px1)*(py2-py1) + (gx2-gx1)*(gy2-gy1) - inter
    return inter/union if union > 0 else 0.0

model.eval()
total_iou, n = 0.0, 0

with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(device)).cpu().numpy()
        for pred, gt in zip(preds, labels.numpy()):
            total_iou += compute_iou(pred[1:5], gt[1:5])
            n += 1

print(f'Mean IoU over {n} validation samples: {total_iou/n:.4f}')

## 6. Inference & Visualisation

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

IDX = 0  # change to inspect a different validation image

img_tensor, gt_label = val_ds[IDX]

model.eval()
with torch.no_grad():
    pred = model(img_tensor.unsqueeze(0).to(device)).cpu().numpy()[0]

print('Ground truth (YOLO):', gt_label.numpy())
print('Prediction   (YOLO):', pred)

# Convert to pixel coords for drawing
H = W = 640

def yolo_to_px(cx, cy, bw, bh, W, H):
    x1 = int((cx - bw/2) * W)
    y1 = int((cy - bh/2) * H)
    x2 = int((cx + bw/2) * W)
    y2 = int((cy + bh/2) * H)
    return x1, y1, x2, y2

img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8).copy()

# Ground truth — red
_, gcx, gcy, gw, gh = gt_label.numpy()
gx1,gy1,gx2,gy2 = yolo_to_px(gcx, gcy, gw, gh, W, H)
cv2.rectangle(img_np, (gx1,gy1), (gx2,gy2), (255,0,0), 2)

# Prediction — green
_, pcx, pcy, pw, ph, conf = pred
px1,py1,px2,py2 = yolo_to_px(pcx, pcy, pw, ph, W, H)
cv2.rectangle(img_np, (px1,py1), (px2,py2), (0,255,0), 2)

plt.figure(figsize=(6, 6))
plt.imshow(img_np)
plt.title(f'Red = Ground Truth  |  Green = Prediction  (conf {conf:.3f})')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f'GT  box (px): ({gx1},{gy1},{gx2},{gy2})')
print(f'Pred box (px): ({px1},{py1},{px2},{py2})')